In [12]:
%pip install fastapi
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph
%pip install -qU langchain-google-genai
%pip install -qU langchain-core
%pip install bs4

I0000 00:00:1753147368.745248 47442663 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1753147369.700085 47442663 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1753147370.824867 47442663 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1753147371.994919 47442663 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1753147373.021258 47442663 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
from fastapi import FastAPI, Query, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import ElasticVectorSearch
import numpy as np
from PIL import Image
import cv2

In [15]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:80"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

llm = init_chat_model("gemini-2.0-flash", model_provider="google_genai")
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

vector_store = ElasticVectorSearch(
    elasticsearch_url="http://localhost:9200",
    index_name="products",
    embedding=embeddings,
)

/Users/manntalati/Documents/Projects/snap-scout-shop/virtualenv/lib/python3.12/site-packages/langchain_community/vectorstores/elastic_vector_search.py:148: UserWarning: ElasticVectorSearch will be removed in a future release. SeeElasticsearch integration docs on how to upgrade.
  warnings.warn(


In [16]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(k=5),
    return_source_documents=True
)

In [17]:
class QueryIn(BaseModel):
    question: str

class ProductDetectionResponse(BaseModel):
    name: str
    brand: str
    price: float
    confidence: float
    category: str

class ChatMessage(BaseModel):
    message: str
    product_data: dict | None = None